In [ ]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
db_path = Path("../../data/database.sqlite")
conn = sqlite3.connect(db_path.as_posix())
df = pd.read_sql_query("SELECT * FROM Match_Aggregated", conn)
conn.close()

In [ ]:
df.info()

# Extract Player Stats

In [ ]:
player_stats_cols = [
    "preferred_foot",
    "attacking_work_rate",
    "defensive_work_rate",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",
    "dribbling",
    "curve",
    "free_kick_accuracy",
    "long_passing",
    "ball_control",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "marking",
    "standing_tackle",
    "sliding_tackle",
]

# Create a copy to avoid SettingWithCopy warnings
df_processed = df.copy()

# Scale work rates from [0, 2] to [-1, 1]
df_processed[["attacking_work_rate", "defensive_work_rate"]] = (
    df_processed[["attacking_work_rate", "defensive_work_rate"]] - 1
)

# Scale attributes from [0, 100] to [-1, 1]
scale_cols = player_stats_cols[3:]
df_processed[scale_cols] = (df_processed[scale_cols] / 50) - 1

In [ ]:
formation_data_list = []

print("Processing 10-player formations (outfield only)...")

# Group by match and team
for (match_id, team_side), df_group in df_processed.groupby(
    ["match_api_id", "team_side"]
):
    # Since you ignore the GK, you expect 10 players.
    # We'll allow a small buffer just in case of red cards/missing data.
    if len(df_group) < 10:
        continue

    # Extract positions and stats
    np_pos = df_group[["player_x", "player_y"]].to_numpy()
    np_stats = df_group[player_stats_cols].to_numpy()

    # Normalize Positions globally (x: 1-9, y: 3-11)
    np_pos_norm = np_pos.copy().astype(float)
    np_pos_norm[:, 0] = (np_pos_norm[:, 0] - 5.0) / 4.0
    np_pos_norm[:, 1] = (np_pos_norm[:, 1] - 7.0) / 4.0

    # Process each of the 10 players
    for i in range(len(df_group)):
        current_player_stats = np_stats[i]
        current_player_pos = np_pos_norm[i]

        # Get relative positions of the other 9 teammates
        teammates_pos = np.delete(np_pos_norm, i, axis=0)

        # If there are exactly 10 players, teammates_pos will be exactly 9.
        # If there are more (e.g. 11), we take the 9 closest.
        if len(teammates_pos) > 9:
            diffs = teammates_pos - current_player_pos
            dists = np.linalg.norm(diffs, axis=1)
            closest_idx = np.argsort(dists)[:9]
            teammates_pos = teammates_pos[closest_idx]

        relative_vectors = teammates_pos - current_player_pos

        # Calculate Distances and Angles
        distances = np.linalg.norm(relative_vectors, ord=2, axis=1) / 2.5
        angles_rad = np.arctan2(relative_vectors[:, 1], relative_vectors[:, 0]) / np.pi

        # Combine: Stats + Global Pos + Distances + Angles
        row = np.hstack(
            [current_player_stats, current_player_pos, distances, angles_rad]
        )
        formation_data_list.append(row)

Processing 10-player formations (outfield only)...


In [ ]:
formation_cols = (
    ["x", "y"]
    + [f"distance_{i + 1}" for i in range(9)]
    + [f"angle_{i + 1}" for i in range(9)]
)

df_final = pd.DataFrame(formation_data_list, columns=player_stats_cols + formation_cols)

In [ ]:
df_final.shape

(427220, 51)

In [ ]:
df_final.head()

,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,finishing,heading_accuracy,short_passing,volleys,dribbling,curve,...,distance_9,angle_1,angle_2,angle_3,angle_4,angle_5,angle_6,angle_7,angle_8,angle_9
0,1.0,0.0,1.0,0.48,-0.14,0.20,0.30,-0.60,0.14,0.30,...,0.400000,0.000000,0.250000,0.187167,0.352416,0.000000,0.500000,0.411414,0.334751,0.000000
1,0.0,0.0,0.0,0.32,0.18,-0.36,0.46,0.34,0.28,0.40,...,0.200000,1.000000,0.647584,0.500000,0.750000,1.000000,0.812833,0.665249,0.588586,1.000000
2,0.0,0.0,1.0,0.44,0.26,-0.24,0.26,-0.16,0.38,-0.04,...,0.400000,-0.750000,-0.352416,0.000000,1.000000,-0.647584,1.000000,0.687167,0.500000,-0.500000
3,1.0,0.0,0.0,0.40,0.08,0.20,0.60,0.18,0.28,0.32,...,0.447214,-0.812833,-0.500000,1.000000,1.000000,-0.750000,1.000000,0.795167,0.687167,-0.647584
4,0.0,-1.0,0.0,0.48,-0.14,0.04,0.76,-0.04,-0.16,0.36,...,0.447214,-0.647584,-0.250000,0.000000,0.000000,-0.500000,1.000000,0.500000,0.312833,-0.352416


In [ ]:
conn = sqlite3.connect(db_path.as_posix())
df_final.to_sql("Player_Formation_Preprocessed", conn, if_exists="replace", index=False)
conn.close()